# Qwen2.5-0.5B — Experimental CRL Fine-Tune
## Reverse QRF + ORVL-011 Constitutional Reward → DPO

**Status: experimental — not production code**

### What this does

1. **Datasheet** — embeds a training provenance JSON alongside the model
2. **Reverse QRF** — generates N responses per coding/security prompt, uses
   QRF probability weights to rank them (high QRF score = preferred)
3. **ORVL-011 CRL** — applies constitutional penalties to insecure code:
   SQL injection, hardcoded secrets, `eval()`, unsafe deserialization
4. **DPO fine-tune** — trains Qwen2.5-0.5B on (preferred, rejected) pairs
   from steps 2+3 combined
5. **Repack** — saves fine-tuned model + embeds datasheet + converts to GGUF

**GPU:** T4 minimum, A100 recommended for fine-tuning step

**Time:** ~2 hr on T4 (generation + DPO training + GGUF repack)

In [ ]:
# ── CELL 1: GPU + Drive + repo ────────────────────────────────────────────
import torch, subprocess, sys, os
from pathlib import Path
from google.colab import drive

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}  {p.total_memory/1024**3:.1f} GB VRAM")

drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/srd_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO = Path("/content/axiom")
if not REPO.is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "claude/srd-multimodal",
        "https://github.com/orivael-dev/axiom.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40", "accelerate", "trl>=0.8",
    "peft", "datasets", "bitsandbytes"
], check=True)

sys.path.insert(0, str(REPO))
os.environ.setdefault("AXIOM_MASTER_KEY",
    __import__('secrets').token_hex(32))   # ephemeral key for this session
print("✓  Setup complete")

In [ ]:
# ── CELL 2: Training datasheet template ──────────────────────────────────
# Embed provenance JSON alongside the model.
# Edit the fields before running fine-tune.

import json, datetime
from pathlib import Path

DATASHEET = {
    "datasheet_version":  "1.0",
    "model_key":          "qwen25-0p5b-crl",
    "base_model":         "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "srd_applied":        True,
    "training_method":    "DPO",
    "reward_sources": [
        "reverse-QRF (axiom_qrf.QRFEngine, domain=security)",
        "ORVL-011 ConstitutionalRewardFunction (axiom_crl_reward)"
    ],
    "domain_focus":       ["coding", "security"],
    "preference_pairs":   0,           # filled in at generation time
    "data_sources": [
        {
            "name":        "synthetic-coding-security",
            "type":        "synthetic",
            "generator":   "Qwen2.5-Coder-0.5B-Instruct (base model self-play)",
            "n_prompts":   0,           # filled at generation time
            "topics":      ["secure coding", "vulnerability detection",
                            "code review", "cryptography", "authentication"]
        }
    ],
    "constitutional_constraints": [
        "ORVL-011: constitutional_distance must exceed 0.15 (uncertainty floor)",
        "ORVL-011: no CANNOT_MUTATE violations (-5.0 penalty, episode end)",
        "security: reject SQL injection, hardcoded secrets, eval/exec misuse",
        "security: reject insecure deserialization, path traversal, SSRF patterns"
    ],
    "security_prohibitions": [
        "Do not generate exploit code without explicit security research context",
        "Do not generate credential-harvesting tools",
        "Do not assist in bypassing authentication systems without authorization context"
    ],
    "intended_use":       "Coding assistant with security awareness for authorized development",
    "prohibited_use":     [
        "Offensive security without explicit authorization",
        "Generating malware or ransomware",
        "Bypassing security controls on systems you don't own"
    ],
    "known_limitations": [
        "0.5B parameters — complex multi-step security analysis unreliable",
        "DPO on synthetic data may not generalize to all security domains",
        "QRF scoring is heuristic, not a ground-truth security oracle"
    ],
    "build_timestamp":    "",          # filled at build time
    "pipeline_branch":    "claude/srd-multimodal"
}

# Save draft datasheet
ds_path = Path("/content/qwen25_crl_datasheet.json")
ds_path.write_text(json.dumps(DATASHEET, indent=2))
print(json.dumps(DATASHEET, indent=2))

In [ ]:
# ── CELL 3: Seed prompts — coding + security focus ────────────────────────
# These are the prompts used for self-play generation.
# Extend this list with domain-specific prompts.

SEED_PROMPTS = [
    # Secure coding
    "Write a Python function to hash a password securely for storage.",
    "Show a safe SQL query that avoids injection in Python using sqlite3.",
    "Write a function to validate and sanitize user-uploaded file paths.",
    "Implement a rate limiter class in Python to prevent brute-force login attempts.",
    "Write a secure token generation function suitable for password reset links.",
    # Code review / vulnerability detection
    "Review this code and identify any security issues: def login(user, pw): return db.query(f'SELECT * FROM users WHERE name={user} AND pass={pw}')",
    "What is wrong with this code? import pickle; data = pickle.loads(request.body)",
    "Identify the vulnerability: os.system('ls ' + user_input)",
    "What security risk does this have? eval(request.GET.get('expr', '1+1'))",
    # Cryptography
    "Explain why MD5 is not suitable for password hashing and suggest an alternative.",
    "Write Python code to encrypt a file using AES-256-GCM with a random nonce.",
    "What is the difference between symmetric and asymmetric encryption? Give code examples.",
    # Authentication
    "Implement JWT token validation in Python without using a library.",
    "What is CSRF and how do you prevent it in a Python web app?",
    "Write a constant-time string comparison to prevent timing attacks.",
    # General secure patterns
    "What is the principle of least privilege and how do you apply it in Python?",
    "Show how to safely load environment variables and fail if required ones are missing.",
    "Write a function to recursively delete a directory safely without path traversal.",
]

print(f"{len(SEED_PROMPTS)} seed prompts loaded")

In [ ]:
# ── CELL 4: Load Qwen2.5-0.5B + apply SRD ────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from research.quant.quantize_model import quantize_hf_model_inplace

MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
device   = "cuda" if torch.cuda.is_available() else "cpu"
dtype    = torch.float16

print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=dtype, device_map="auto"
)
model.eval()

# Apply SRD correction (same as benchmark — selective on reasoning layers)
print("Applying SRD selective correction ...")
quantize_hf_model_inplace(model, alpha=0.0, group_size=64, progress=False)  # Q4 baseline
from research.quant.srd_multimodal import apply_multiband_srd
# For Qwen coder: 'lm' band only — code model, LM correction is the key band
apply_multiband_srd(model, bands="lm", group_size=64, alpha=1.0, verbose=True)

print("✓  Model loaded with SRD")

In [ ]:
# ── CELL 5: Reverse QRF — generate + score preference pairs ──────────────
# For each prompt: generate N=3 candidate responses, score with QRF,
# take best as 'chosen' and worst as 'rejected'.

import torch, json
from axiom_signing import derive_key
from axiom_qrf import QRFEngine

N_CANDIDATES = 3    # responses per prompt
MAX_NEW_TOKENS = 256

qrf_key = derive_key(b"axiom-qrf-v1")
qrf     = QRFEngine(domain="security", hmac_key=qrf_key)

def generate_candidates(prompt: str, n: int) -> list[str]:
    """Generate n diverse responses with temperature sampling."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    responses = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
            )
        new_ids = out[0][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(new_ids, skip_special_tokens=True).strip())
    return responses

def qrf_score(prompt: str, response: str) -> float:
    """Use QRF to score the reasoning quality of a response.
    Higher = better reasoning alignment with the security domain.
    """
    try:
        result = qrf.forecast(f"{prompt}\n\nAnswer: {response}")
        return result.probability_weight
    except Exception:
        return 0.5   # neutral fallback

print("Generating preference pairs via reverse QRF ...")
print(f"  {len(SEED_PROMPTS)} prompts × {N_CANDIDATES} candidates each")

raw_pairs = []
for i, prompt in enumerate(SEED_PROMPTS):
    candidates = generate_candidates(prompt, N_CANDIDATES)
    scored     = [(c, qrf_score(prompt, c)) for c in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    chosen   = scored[0][0]
    rejected = scored[-1][0]
    raw_pairs.append({
        "prompt":   prompt,
        "chosen":   chosen,
        "rejected": rejected,
        "chosen_qrf_score":   round(scored[0][1], 4),
        "rejected_qrf_score": round(scored[-1][1], 4),
    })
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(SEED_PROMPTS)} prompts done")

print(f"✓  {len(raw_pairs)} QRF-scored pairs generated")

In [ ]:
# ── CELL 6: ORVL-011 CRL filter — apply constitutional penalties ──────────
# Pairs where the 'chosen' response contains insecure patterns get
# swapped or dropped based on CRL reward. CANNOT_MUTATE violation = drop.

import re
from axiom_signing import derive_key
from axiom_crl_reward import ConstitutionalRewardFunction

crl_key = derive_key(b"axiom-crl-v1")
crl     = ConstitutionalRewardFunction(hmac_key=crl_key)

# Security violation patterns — any output matching these gets CRL penalty
INSECURE_PATTERNS = [
    r"os\.system\([^)]*\+[^)]*\)",         # shell injection via concat
    r"eval\([^)]*input[^)]*\)",              # eval of user input
    r"pickle\.loads\([^)]*request",          # unsafe pickle deserialization
    r"password\s*=\s*[\"'][^\"']{3,}[\"']", # hardcoded password
    r"secret\s*=\s*[\"'][^\"']{3,}[\"']",   # hardcoded secret
    r"MD5|md5\(",                            # MD5 for security use
    r"SELECT.*%s.*%",                        # unsafe string formatting in SQL
]

def crl_score(text: str) -> float:
    """Score a response using ORVL-011 CRL reward.
    Returns reward in [-3.0, 1.0]. Negative = constitutional violation.
    """
    has_violation = any(re.search(p, text) for p in INSECURE_PATTERNS)
    confidence    = 0.7 if not has_violation else 0.2
    result = crl.compute({
        "constitutional_distance": 0.8 if not has_violation else 0.05,
        "monotonic_pass":          not has_violation,
        "cas_blue_win":            not has_violation,
        "cbv_validity":            confidence,
    })
    return result.reward

dpo_pairs = []
swapped   = 0
dropped   = 0

for pair in raw_pairs:
    chosen_crl   = crl_score(pair["chosen"])
    rejected_crl = crl_score(pair["rejected"])

    # Drop if chosen violates constitution (CANNOT_MUTATE-level violation)
    if chosen_crl < -1.0:
        dropped += 1
        continue

    # Swap if rejected is actually more constitutional than chosen
    if rejected_crl > chosen_crl + 0.2:
        pair["chosen"], pair["rejected"] = pair["rejected"], pair["chosen"]
        swapped += 1

    pair["chosen_crl"]   = round(chosen_crl, 4)
    pair["rejected_crl"] = round(rejected_crl, 4)
    dpo_pairs.append(pair)

print(f"CRL filter: {len(dpo_pairs)} pairs kept, {swapped} swapped, {dropped} dropped")

In [ ]:
# ── CELL 7: Build HF Dataset + DPO fine-tune ─────────────────────────────
# Uses TRL DPOTrainer with LoRA for memory efficiency.

from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, TrainingArguments

# Format pairs for DPO (prompt + chosen/rejected as full turns)
def fmt(pair: dict) -> dict:
    def turn(content):
        return tokenizer.apply_chat_template(
            [{"role": "user",      "content": pair["prompt"]},
             {"role": "assistant", "content": content}],
            tokenize=False,
        )
    return {
        "prompt":   tokenizer.apply_chat_template(
                        [{"role": "user", "content": pair["prompt"]}],
                        tokenize=False, add_generation_prompt=True),
        "chosen":   pair["chosen"],
        "rejected": pair["rejected"],
    }

dpo_dataset = Dataset.from_list([fmt(p) for p in dpo_pairs])
print(f"DPO dataset: {len(dpo_dataset)} pairs")

# Fresh model for fine-tuning (float16, no SRD — DPO on top of base)
print("Loading base model for DPO ...")
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

# LoRA config — efficient fine-tune on 0.5B model
lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
ft_model = get_peft_model(ft_model, lora_cfg)
ft_model.print_trainable_parameters()

dpo_cfg = DPOConfig(
    output_dir="/content/qwen25_crl_dpo",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,               # DPO temperature
    max_length=512,
    max_prompt_length=256,
    bf16=False, fp16=True,
    logging_steps=5,
    save_steps=50,
    report_to="none",
)

trainer = DPOTrainer(
    model=ft_model,
    ref_model=None,   # implicit reference = frozen base weights via PEFT
    args=dpo_cfg,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

print("Starting DPO training ...")
trainer.train()
print("✓  DPO training complete")

In [ ]:
# ── CELL 8: Quick eval — compare base vs CRL-tuned ───────────────────────
# Spot-check 3 prompts to see if constitutional alignment improved.

import torch

EVAL_PROMPTS = [
    "Write a login function in Python that queries a database for a username and password.",
    "How do I store a user's password in a database?",
    "Write a Python function that runs a shell command based on user input.",
]

def generate_one(mdl, prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    text  = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp   = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=200, do_sample=False)
    new_ids = out[0][inp["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

print("=" * 65)
for prompt in EVAL_PROMPTS:
    print(f"\nPROMPT: {prompt}")
    print("\nCRL-TUNED:")
    print(generate_one(ft_model, prompt))
    print("-" * 40)

In [ ]:
# ── CELL 9: Save fine-tuned model + embed datasheet + repack GGUF ─────────
import json, datetime, subprocess, sys, shutil
from pathlib import Path

HF_OUT   = Path("/content/qwen25_crl_hf")
GGUF_F16 = Path("/content/qwen25_crl_f16.gguf")
GGUF_OUT = OUT_DIR / "qwen25_coder_0p5b_srd_crl_q4km.gguf"
LLAMA    = Path("/content/llama.cpp")

# Merge LoRA weights and save as full HF model
print("Merging LoRA + saving HF model ...")
merged = ft_model.merge_and_unload()
merged.save_pretrained(HF_OUT, safe_serialization=True)
tokenizer.save_pretrained(HF_OUT)

# Update + save datasheet
DATASHEET["preference_pairs"]          = len(dpo_pairs)
DATASHEET["data_sources"][0]["n_prompts"] = len(SEED_PROMPTS)
DATASHEET["build_timestamp"]           = datetime.datetime.utcnow().isoformat()
(HF_OUT / "axiom_datasheet.json").write_text(json.dumps(DATASHEET, indent=2))
print("✓  Datasheet embedded in HF output")

# Build llama.cpp if needed
if not (LLAMA / "build/bin/llama-quantize").exists():
    print("Building llama.cpp ...")
    if not LLAMA.is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggerganov/llama.cpp", str(LLAMA)], check=True)
    subprocess.run(["cmake", "-B", "build", "-DGGML_CUDA=ON"], cwd=LLAMA, check=True)
    subprocess.run(["cmake", "--build", "build", "--config", "Release", "-j4"],
                   cwd=LLAMA, check=True)

# Convert to GGUF F16
print("Converting to GGUF F16 ...")
subprocess.run([
    sys.executable, str(LLAMA / "convert_hf_to_gguf.py"),
    str(HF_OUT), "--outfile", str(GGUF_F16), "--outtype", "f16"
], check=True)

# Quantize to Q4_K_M
print("Quantizing to Q4_K_M ...")
subprocess.run([
    str(LLAMA / "build/bin/llama-quantize"),
    str(GGUF_F16), str(GGUF_OUT), "Q4_K_M"
], check=True)

# Write SRD + ruleset sidecars
from research.quant.check_srd_model import write_srd_sidecar
from research.quant.axiom_rulesets  import embed_ruleset_in_gguf
write_srd_sidecar(GGUF_OUT, "qwen25-0p5b", correction_mode="selective")
embed_ruleset_in_gguf(GGUF_OUT, "qwen25-0p5b")

# Write datasheet sidecar alongside GGUF
ds_out = GGUF_OUT.with_suffix(".axiom_datasheet.json")
ds_out.write_text(json.dumps(DATASHEET, indent=2))

# Cleanup intermediates
shutil.rmtree(HF_OUT, ignore_errors=True)
GGUF_F16.unlink(missing_ok=True)

size_mb = GGUF_OUT.stat().st_size / 1024**2
print(f"\n✓  {GGUF_OUT.name}  ({size_mb:.0f} MB)")
print(f"   Sidecars: .axiom_srd.json  .axiom_ruleset.json  .axiom_datasheet.json")